# Actividad 4 — Quiz 1: Tokenizador de subpalabras con SentencePiece

**Maestría Virtual en Ingeniería de Sistemas y Computación — PLN con Deep Learning**

**Autor:** Raúl Andrés Alzate Gómez  
**Dataset:** [`jhonparra18/spanish_billion_words_clean`](https://huggingface.co/datasets/jhonparra18/spanish_billion_words_clean) (Hugging Face)

## Objetivo
Realizar varios testeos del algoritmo **SentencePiece** sobre el dataset *Word Billion* en español, comparando los modelos `bpe`, `unigram` (default), `word` y `char` con vocabularios de 10 000, 20 000 y 30 000 tokens, e incluyendo símbolos de control al estilo WordPiece (`<sep>`, `<cls>`).


> Nota: el dataset oficial *Word Billion* tiene ~47 M de oraciones (≈4.7 GB). Para que el notebook sea reproducible en hardware modesto se trabaja con un **subconjunto** (configurable mediante `MAX_SENTENCES`); la rutina de particionado y el flujo de entrenamiento son idénticos al caso de full dataset, basta con incrementar la constante.

## 1. Instalación de librerías

`datasets` (Hugging Face) para cargar el corpus en streaming y `sentencepiece` (Google) como motor de tokenización de subpalabras.

In [1]:
%pip install -q datasets sentencepiece tqdm


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, glob, json, shutil, pathlib, itertools
from tqdm.auto import tqdm
import sentencepiece as spm
from datasets import load_dataset

print('sentencepiece', spm.__version__)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sentencepiece 0.2.1


## 2. Carga del dataset Word Billion (Hugging Face)

Como el dataset pesa ~4.7 GB lo cargamos en **modo streaming** (`streaming=True`) — no se descarga completo, se itera registro a registro. La constante `MAX_SENTENCES` controla cuántas frases se materializan en disco; subirla acerca el experimento al *full dataset* sin tocar el resto del código.

In [3]:
DATASET_NAME = 'jhonparra18/spanish_billion_words_clean'
MAX_SENTENCES = 200_000   # subir a None (o 47_000_000) para usar el dataset completo
CHUNK_SIZE    = 5_000     # frases por archivo de texto

DATA_DIR = pathlib.Path('./word_billion_chunks')
DATA_DIR.mkdir(exist_ok=True)

ds = load_dataset(DATASET_NAME, split='train', streaming=True)
print('Dataset cargado en streaming:', ds)

Dataset cargado en streaming: IterableDataset({
    features: ['text'],
    num_shards: 18
})


## 3. Particionado del dataset en archivos de texto

SentencePiece consume **archivos de texto plano** (uno o varios). Para datasets masivos como Word Billion la práctica estándar es trocearlo en chunks de N frases — esto permite (a) controlar memoria, (b) entrenar sobre un subset balanceado, y (c) habilitar entrenamiento incremental.

La rutina sigue el patrón sugerido en el enunciado: acumular en `text_data`, escribir cada `CHUNK_SIZE` frases a un nuevo archivo `text_{i}.txt`, y volcar el remanente al final.

In [4]:
def particionar_dataset(ds_iter, out_dir: pathlib.Path, chunk_size: int, max_sentences):
    """Vuelca un iterador de Hugging Face en archivos text_{i}.txt de chunk_size líneas."""
    out_dir.mkdir(exist_ok=True)
    for f in out_dir.glob('text_*.txt'):
        f.unlink()

    text_data, file_count, total = [], 0, 0
    bar = tqdm(total=max_sentences, desc='Particionando')
    for sample in ds_iter:
        sentence = sample['text'].replace('\n', ' ').strip()
        if not sentence:
            continue
        text_data.append(sentence)
        total += 1
        bar.update(1)
        if len(text_data) == chunk_size:
            (out_dir / f'text_{file_count}.txt').write_text('\n'.join(text_data), encoding='utf-8')
            text_data, file_count = [], file_count + 1
        if max_sentences and total >= max_sentences:
            break
    if text_data:
        (out_dir / f'text_{file_count}.txt').write_text('\n'.join(text_data), encoding='utf-8')
        file_count += 1
    bar.close()
    return file_count, total

n_files, n_sentences = particionar_dataset(ds, DATA_DIR, CHUNK_SIZE, MAX_SENTENCES)
print(f'Generados {n_files} archivos con {n_sentences:,} oraciones en {DATA_DIR}/')

Particionando: 100%|██████████| 200000/200000 [00:08<00:00, 22806.41it/s]

Generados 40 archivos con 200,000 oraciones en word_billion_chunks/


In [5]:
# Verificación — listar archivos producidos y mostrar muestra
txt_files = sorted(DATA_DIR.glob('text_*.txt'))
print('Archivos:', [f.name for f in txt_files[:5]], '...')
print('Total archivos:', len(txt_files))
print('\nPrimeras líneas del primer archivo:')
print('\n'.join(txt_files[0].read_text(encoding='utf-8').splitlines()[:5]))

Archivos: ['text_0.txt', 'text_1.txt', 'text_10.txt', 'text_11.txt', 'text_12.txt'] ...
Total archivos: 40

Primeras líneas del primer archivo:
source wikisource librodot com sentido y sensibilidad jane austen capitulo i
la familia dashwood llevaba largo tiempo afincada en sussex
su propiedad era de buen tamaño y en el centro de ella se encontraba la residencia norland park donde la manera tan digna en que habían vivido por muchas generaciones llegó a granjearles el respeto de todos los conocidos del lugar
el último dueño de esta propiedad había sido un hombre soltero que alcanzó una muy avanzada edad y que durante gran parte de su existencia tuvo en su hermana una fiel compañera y ama de casa
pero la muerte de ella ocurrida diez años antes que la suya produjo grandes alteraciones en su hogar


## 4. Entrenamiento base de SentencePiece (`model_prefix='m'`)

Primer entrenamiento con la configuración por defecto (modelo Unigram) sobre el conjunto completo de archivos de texto. El parámetro `input` admite una lista separada por comas de rutas de archivo. El parámetro `model_prefix='m'` produce los artefactos `m.model` (modelo binario) y `m.vocab` (vocabulario legible).

> `input_sentence_size` y `shuffle_input_sentence` son cruciales cuando se trabaja con corpus muy grandes — limitan cuántas frases ve el entrenador y las muestrean aleatoriamente, evitando OOM.

In [6]:
MODELS_DIR = pathlib.Path('./models')
MODELS_DIR.mkdir(exist_ok=True)

input_files = ','.join(str(f) for f in txt_files)

spm.SentencePieceTrainer.train(
    input=input_files,
    model_prefix=str(MODELS_DIR / 'm'),
    vocab_size=20000,
    character_coverage=1.0,
    input_sentence_size=200_000,
    shuffle_input_sentence=True,
)

print('Artefactos generados:')
for f in sorted(MODELS_DIR.glob('m.*')):
    print(' -', f, f'({f.stat().st_size/1024:.1f} KB)')

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: word_billion_chunks/text_0.txt
  input: word_billion_chunks/text_1.txt
  input: word_billion_chunks/text_10.txt
  input: word_billion_chunks/text_11.txt
  input: word_billion_chunks/text_12.txt
  input: word_billion_chunks/text_13.txt
  input: word_billion_chunks/text_14.txt
  input: word_billion_chunks/text_15.txt
  input: word_billion_chunks/text_16.txt
  input: word_billion_chunks/text_17.txt
  input: word_billion_chunks/text_18.txt
  input: word_billion_chunks/text_19.txt
  input: word_billion_chunks/text_2.txt
  input: word_billion_chunks/text_20.txt
  input: word_billion_chunks/text_21.txt
  input: word_billion_chunks/text_22.txt
  input: word_billion_chunks/text_23.txt
  input: word_billion_chunks/text_24.txt
  input: word_billion_chunks/text_25.txt
  input: word_billion_chunks/text_26.txt
  input: word_billion_chunks/text_27.txt
  input: word_billion_chunks/text_28.txt
  input: word_billion_c

Artefactos generados:
 - models/m.model (602.3 KB)
 - models/m.vocab (383.8 KB)


ord_billion_chunks/text_6.txt
trainer_interface.cc(186) LOG(INFO) Loading corpus: word_billion_chunks/text_7.txt
trainer_interface.cc(186) LOG(INFO) Loading corpus: word_billion_chunks/text_8.txt
trainer_interface.cc(186) LOG(INFO) Loading corpus: word_billion_chunks/text_9.txt
trainer_interface.cc(411) LOG(INFO) Loaded all 200000 sentences
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <unk>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <s>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: </s>
trainer_interface.cc(432) LOG(INFO) Normalizing sentences...
trainer_interface.cc(541) LOG(INFO) all chars count=27347773
trainer_interface.cc(562) LOG(INFO) Alphabet size=34
trainer_interface.cc(563) LOG(INFO) Final character coverage=1
trainer_interface.cc(594) LOG(INFO) Done! preprocessed 200000 sentences.
unigram_model_trainer.cc(265) LOG(INFO) Making suffix array...
unigram_model_trainer.cc(269) LOG(INFO) Extracting frequent sub strings... node_num=15059602
unigram_mo

In [7]:
# Carga del modelo base y prueba
sp_base = spm.SentencePieceProcessor()
sp_base.load(str(MODELS_DIR / 'm.model'))

demo = 'El procesamiento de lenguaje natural revoluciona la inteligencia artificial.'
print('Pieces:', sp_base.encode_as_pieces(demo))
print('IDs   :', sp_base.encode_as_ids(demo))
print('Decode:', sp_base.decode_pieces(sp_base.encode_as_pieces(demo)))

Pieces: ['▁', 'E', 'l', '▁procesamiento', '▁de', '▁lenguaje', '▁natural', '▁revolucion', 'a', '▁la', '▁inteligencia', '▁artificial', '.']
IDs   : [228, 0, 123, 5598, 3, 2486, 465, 13721, 39, 5, 7580, 5241, 0]
Decode: El procesamiento de lenguaje natural revoluciona la inteligencia artificial.


## 5. Tokenización de 5 sentencias con distintos tipos de modelo

Se definen las 5 oraciones de prueba y una función auxiliar para entrenar / evaluar de forma uniforme cada configuración.

In [8]:
SENTENCIAS = [
    'El procesamiento de lenguaje natural revoluciona la inteligencia artificial.',
    'Los tokenizadores de subpalabras manejan vocabulario abierto sin perder información.',
    'SentencePiece entrena directamente sobre texto en bruto sin pretokenización.',
    'Las redes neuronales transformer usan atención multi-cabeza para modelar contexto.',
    'La universidad ofrece la maestría en ingeniería de sistemas y computación.',
]

def entrenar_y_tokenizar(model_prefix: str, model_type: str, vocab_size: int,
                         user_defined_symbols=None, control_symbols=None,
                         **extra):
    """Entrena un modelo SentencePiece y tokeniza SENTENCIAS. Devuelve resultados."""
    out_prefix = MODELS_DIR / model_prefix
    kwargs = dict(
        input=input_files,
        model_prefix=str(out_prefix),
        vocab_size=vocab_size,
        model_type=model_type,
        character_coverage=1.0,
        input_sentence_size=200_000,
        shuffle_input_sentence=True,
    )
    if user_defined_symbols:
        kwargs['user_defined_symbols'] = ','.join(user_defined_symbols)
    if control_symbols:
        kwargs['control_symbols'] = ','.join(control_symbols)
    kwargs.update(extra)

    spm.SentencePieceTrainer.train(**kwargs)
    sp = spm.SentencePieceProcessor()
    sp.load(str(out_prefix) + '.model')

    resultados = []
    for s in SENTENCIAS:
        pieces = sp.encode_as_pieces(s)
        ids    = sp.encode_as_ids(s)
        resultados.append({'sentencia': s, 'pieces': pieces, 'ids': ids, 'n_tokens': len(pieces)})
    return sp, resultados

def imprimir(resultados, titulo):
    print(f'\n=== {titulo} ===')
    for i, r in enumerate(resultados, 1):
        print(f'\n[{i}] ({r["n_tokens"]} tokens) {r["sentencia"]}')
        print('   pieces:', r['pieces'])
        print('   ids   :', r['ids'])

### 5.1 Modelo BPE — vocabularios 10 000, 20 000, 30 000

**Byte-Pair Encoding (BPE)** [Sennrich et al., 2016] parte de un vocabulario de caracteres y fusiona iterativamente los pares más frecuentes hasta alcanzar el tamaño objetivo. Útil cuando se busca un balance compresión/cobertura.

In [15]:
modelos_bpe = {}
for vs in (10000, 20000, 30000):
    sp, res = entrenar_y_tokenizar(model_prefix=f'bpe_{vs}', model_type='bpe', vocab_size=vs)
    modelos_bpe[vs] = (sp, res)
    imprimir(res, f'BPE — vocab_size={vs}')


=== BPE — vocab_size=10000 ===

[1] (15 tokens) El procesamiento de lenguaje natural revoluciona la inteligencia artificial.
   pieces: ['▁', 'E', 'l', '▁procesamiento', '▁de', '▁lenguaje', '▁natural', '▁revolu', 'cion', 'a', '▁la', '▁intelig', 'encia', '▁artificial', '.']
   ids   : [9966, 0, 9976, 9200, 5, 5747, 1087, 8901, 71, 9968, 37, 5369, 356, 9699, 0]

[2] (23 tokens) Los tokenizadores de subpalabras manejan vocabulario abierto sin perder información.
   pieces: ['▁', 'L', 'os', '▁to', 'ken', 'iz', 'adores', '▁de', '▁sub', 'pal', 'ab', 'ras', '▁mane', 'jan', '▁vo', 'cab', 'ul', 'ario', '▁abierto', '▁sin', '▁perder', '▁información', '.']
   ids   : [9966, 0, 17, 88, 5767, 153, 448, 5, 1024, 3093, 59, 630, 4112, 6087, 1134, 7980, 147, 301, 2386, 292, 4247, 480, 0]

[3] (22 tokens) SentencePiece entrena directamente sobre texto en bruto sin pretokenización.
   pieces: ['▁', 'S', 'en', 'ten', 'ce', 'P', 'i', 'ece', '▁entr', 'ena', '▁directamente', '▁sobre', '▁texto', '▁en', '▁bru'

### 5.2 Símbolos de control estilo WordPiece (`<sep>`, `<cls>`)

WordPiece (BERT) introduce tokens especiales `[CLS]` (clasificación) y `[SEP]` (separador) para tareas downstream. SentencePiece los soporta vía `control_symbols` (no aparecen en la salida del tokenizer pero ocupan ID dedicado) o `user_defined_symbols` (preservados como tokens unitarios cuando aparecen en el texto).

In [17]:
modelos_wp = {}
for vs in (10000, 20000, 30000):
    sp, res = entrenar_y_tokenizar(
        model_prefix=f'wp_{vs}',
        model_type='bpe',
        vocab_size=vs,
        user_defined_symbols=['<sep>', '<cls>'],
    )
    modelos_wp[vs] = (sp, res)
    imprimir(res, f'WordPiece-style (BPE + <sep>/<cls>) — vocab_size={vs}')

# Demostración del manejo de los símbolos de control
sp = modelos_wp[20000][0]
demo = '<cls> El modelo aprende contexto. <sep> Las frases siguen.'
print('\nDemo con <cls>/<sep>:')
print('  pieces:', sp.encode_as_pieces(demo))
print('  ids   :', sp.encode_as_ids(demo))
print('  <cls> id =', sp.piece_to_id('<cls>'), '| <sep> id =', sp.piece_to_id('<sep>'))


=== WordPiece-style (BPE + <sep>/<cls>) — vocab_size=10000 ===

[1] (15 tokens) El procesamiento de lenguaje natural revoluciona la inteligencia artificial.
   pieces: ['▁', 'E', 'l', '▁procesamiento', '▁de', '▁lenguaje', '▁natural', '▁revolu', 'cion', 'a', '▁la', '▁intelig', 'encia', '▁artificial', '.']
   ids   : [9966, 0, 9976, 9202, 7, 5749, 1089, 8903, 73, 9968, 39, 5371, 358, 9701, 0]

[2] (23 tokens) Los tokenizadores de subpalabras manejan vocabulario abierto sin perder información.
   pieces: ['▁', 'L', 'os', '▁to', 'ken', 'iz', 'adores', '▁de', '▁sub', 'pal', 'ab', 'ras', '▁mane', 'jan', '▁vo', 'cab', 'ul', 'ario', '▁abierto', '▁sin', '▁perder', '▁información', '.']
   ids   : [9966, 0, 19, 90, 5769, 155, 450, 7, 1026, 3095, 61, 632, 4114, 6089, 1136, 7982, 149, 303, 2388, 294, 4249, 482, 0]

[3] (22 tokens) SentencePiece entrena directamente sobre texto en bruto sin pretokenización.
   pieces: ['▁', 'S', 'en', 'ten', 'ce', 'P', 'i', 'ece', '▁entr', 'ena', '▁directamente', '

### 5.3 Modelo `word`

Tokenización por **palabras completas**: cada palabra del corpus de entrenamiento se asigna a un ID; lo desconocido cae al token `<unk>`. Es el extremo más simple, sufre con vocabulario abierto pero sirve como baseline.

In [19]:
modelos_word = {}
for vs in (10000, 20000, 30000):
    sp, res = entrenar_y_tokenizar(
        model_prefix=f'word_{vs}',
        model_type='word',
        vocab_size=vs,
    )
    modelos_word[vs] = (sp, res)
    imprimir(res, f'WORD — vocab_size={vs}')


=== WORD — vocab_size=10000 ===

[1] (9 tokens) El procesamiento de lenguaje natural revoluciona la inteligencia artificial.
   pieces: ['▁El', '▁procesamiento', '▁de', '▁lenguaje', '▁natural', '▁revoluciona', '▁la', '▁inteligencia', '▁artificial.']
   ids   : [0, 4336, 3, 2331, 395, 0, 6, 5953, 0]

[2] (7 tokens) Los tokenizadores de subpalabras manejan vocabulario abierto sin perder información.
   pieces: ['▁Los▁tokenizadores', '▁de', '▁subpalabras▁manejan▁vocabulario', '▁abierto', '▁sin', '▁perder', '▁información.']
   ids   : [0, 3, 0, 708, 56, 2276, 0]

[3] (8 tokens) SentencePiece entrena directamente sobre texto en bruto sin pretokenización.
   pieces: ['▁SentencePiece▁entrena', '▁directamente', '▁sobre', '▁texto', '▁en', '▁bruto', '▁sin', '▁pretokenización.']
   ids   : [0, 763, 48, 1015, 8, 0, 56, 0]

[4] (8 tokens) Las redes neuronales transformer usan atención multi-cabeza para modelar contexto.
   pieces: ['▁Las', '▁redes', '▁neuronales▁transformer', '▁usan', '▁atención',

### 5.4 Modelo `char`

Tokenización por **caracteres individuales**. Vocabulario mínimo, secuencias muy largas, cero OOV. SentencePiece exige `vocab_size` ≥ caracteres únicos del corpus; en español suele requerir ≥ 100. Aun así, evaluamos con vocabularios de 10 000 / 20 000 / 30 000 (el modelo solo usa los necesarios; el resto del presupuesto queda sin asignar).

In [21]:
modelos_char = {}
for vs in (10000, 20000, 30000):
    try:
        sp, res = entrenar_y_tokenizar(
            model_prefix=f'char_{vs}',
            model_type='char',
            vocab_size=vs,
        )
    except RuntimeError as e:
        # 'char' suele exigir vocab_size pequeño; reintenta con tamaño efectivo
        print(f'[warn] vocab_size={vs} no aplica para char ({e}); reintentando con 4000.')
        sp, res = entrenar_y_tokenizar(
            model_prefix=f'char_{vs}',
            model_type='char',
            vocab_size=4000,
        )
    modelos_char[vs] = (sp, res)
    imprimir(res, f'CHAR — vocab_size={vs} (efectivo: {sp.get_piece_size()})')


=== CHAR — vocab_size=10000 (efectivo: 37) ===

[1] (77 tokens) El procesamiento de lenguaje natural revoluciona la inteligencia artificial.
   pieces: ['▁', 'E', 'l', '▁', 'p', 'r', 'o', 'c', 'e', 's', 'a', 'm', 'i', 'e', 'n', 't', 'o', '▁', 'd', 'e', '▁', 'l', 'e', 'n', 'g', 'u', 'a', 'j', 'e', '▁', 'n', 'a', 't', 'u', 'r', 'a', 'l', '▁', 'r', 'e', 'v', 'o', 'l', 'u', 'c', 'i', 'o', 'n', 'a', '▁', 'l', 'a', '▁', 'i', 'n', 't', 'e', 'l', 'i', 'g', 'e', 'n', 'c', 'i', 'a', '▁', 'a', 'r', 't', 'i', 'f', 'i', 'c', 'i', 'a', 'l', '.']
   ids   : [3, 0, 13, 3, 17, 10, 6, 14, 4, 8, 5, 16, 7, 4, 9, 11, 6, 3, 12, 4, 3, 13, 4, 9, 18, 15, 5, 27, 4, 3, 9, 5, 11, 15, 10, 5, 13, 3, 10, 4, 22, 6, 13, 15, 14, 7, 6, 9, 5, 3, 13, 5, 3, 7, 9, 11, 4, 13, 7, 18, 4, 9, 14, 7, 5, 3, 5, 10, 11, 7, 23, 7, 14, 7, 5, 13, 0]

[2] (85 tokens) Los tokenizadores de subpalabras manejan vocabulario abierto sin perder información.
   pieces: ['▁', 'L', 'o', 's', '▁', 't', 'o', 'k', 'e', 'n', 'i', 'z', 'a', 'd', 'o',

## 6. Comparativa global y conclusiones

Tabla resumen del **número de tokens** que cada modelo produce sobre las 5 sentencias. Menos tokens = mejor compresión (menor longitud de secuencia para el modelo downstream); pero un vocabulario muy compacto puede perder granularidad morfológica.

In [23]:
def fila(label, resultados):
    return [label] + [r['n_tokens'] for r in resultados] + [sum(r['n_tokens'] for r in resultados)]

header = ['modelo', 'S1', 'S2', 'S3', 'S4', 'S5', 'TOTAL']
rows = []
for vs in (10000, 20000, 30000):
    rows.append(fila(f'bpe-{vs}',  modelos_bpe[vs][1]))
    rows.append(fila(f'wp-{vs}',   modelos_wp[vs][1]))
    rows.append(fila(f'word-{vs}', modelos_word[vs][1]))
    rows.append(fila(f'char-{vs}', modelos_char[vs][1]))

col_w = [max(len(str(r[i])) for r in rows + [header]) for i in range(len(header))]
fmt = '  '.join('{:>' + str(w) + '}' for w in col_w)
print(fmt.format(*header))
print('-' * (sum(col_w) + 2 * (len(header) - 1)))
for r in rows:
    print(fmt.format(*r))

    modelo  S1  S2  S3  S4  S5  TOTAL
-------------------------------------
 bpe-10000  15  23  22  22  18    100
  wp-10000  15  23  22  22  18    100
word-10000   9   7   8   8  11     43
char-10000  77  85  77  83  75    397
 bpe-20000  13  21  22  19  16     91
  wp-20000  13  21  22  19  16     91
word-20000   9   9   8   8  11     45
char-20000  77  85  77  83  75    397
 bpe-30000  13  18  20  19  14     84
  wp-30000  13  18  20  19  14     84
word-30000   9   9   8   8  11     45
char-30000  77  85  77  83  75    397


In [24]:
# Inventario final de modelos generados
print('Modelos guardados en ./models/:')
for f in sorted(MODELS_DIR.glob('*.model')):
    print(f'  {f.name:<25}  {f.stat().st_size/1024:>8.1f} KB')

Modelos guardados en ./models/:
  bpe_10000.model               401.8 KB
  bpe_20000.model               579.7 KB
  bpe_30000.model               760.6 KB
  char_10000.model              236.1 KB
  char_20000.model              236.1 KB
  char_30000.model              236.1 KB
  m.model                       602.3 KB
  word_10000.model              425.2 KB
  word_20000.model              619.0 KB
  word_30000.model              814.6 KB
  wp_10000.model                401.8 KB
  wp_20000.model                579.7 KB
  wp_30000.model                760.6 KB
